# Databricks DE Associate — Hands-On Lab Pack
### Built against the May 2026 exam outline

**How to use this**

1. Import into your workspace: Workspace → (folder) → Import → File → select this `.ipynb`.
2. Attach to **serverless** compute (Free Edition is serverless-only).
3. Run **Lab 0** first — it creates everything the other labs depend on.
4. Labs 1–4 run here. Labs 5–7 are driven from the Jobs UI and a terminal; this notebook holds their code and checklists.

**Time budget:** L1 3h · L2 2h · L3 1.5h · L4 2h · L5 2h · L6 2.5h · L7 1h

**The rule that makes this work:** each lab has *Observe* cells. Don't skip them.
Running code you already understand teaches nothing; the marks are in noticing
what surprised you. Keep the scratch cell at the bottom of each lab for that.

**Caveats:** Free Edition has per-account quotas — if compute dies mid-lab you've
hit the daily cap and it resets tomorrow. Cells flagged **[MAY NOT RUN ON FREE EDITION]**
have a stated fallback. Nothing here writes outside your own schema.

---
# Lab 0 — Setup

Creates a schema and volume, and generates the source files every later lab uses.
Edit `CATALOG` if you aren't using the Free Edition default.

In [0]:
CATALOG = "workspace"     # Free Edition default; change if you have your own
SCHEMA  = "de_labs"
VOLUME  = "landing"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

VOL = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
print("Landing volume:", VOL)

### Generate source files

Three batches of JSON orders. Batch 3 deliberately adds a column that batches 1–2
lack — that's what Lab 1 uses to exercise schema evolution.

In [0]:
import json, random, datetime, os

random.seed(42)
os.makedirs(f"{VOL}/orders", exist_ok=True)

def make_batch(name, n, start_id, with_channel=False, dirty=False):
    rows = []
    for i in range(n):
        r = {
            "order_id": f"o{start_id+i}",
            "customer_id": f"c{random.randint(1,60)}",
            "order_ts": (datetime.datetime(2026,3,1) +
                         datetime.timedelta(minutes=random.randint(0,40000))
                        ).strftime("%Y-%m-%d %H:%M:%S"),
            "amount": f"{random.uniform(5,900):.2f}",
            "region": random.choice(["EMEA","AMER","APAC"]),
            "items": [{"sku": f"s{random.randint(1,25)}",
                       "qty": random.randint(1,4)}
                      for _ in range(random.randint(1,3))],
        }
        if with_channel:
            r["channel"] = random.choice(["web","app","store"])
        if dirty and i % 17 == 0:
            r["amount"] = ""          # empty string -> tests null-safe casting
        rows.append(r)
    with open(f"{VOL}/orders/{name}", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    return len(rows)

print("batch_01:", make_batch("batch_01.json", 400, 1000, dirty=True))
print("batch_02:", make_batch("batch_02.json", 400, 2000, dirty=True))
# batch_03 held back — you land it mid-stream in Lab 1
print("held back: batch_03 (adds 'channel' column)")

In [0]:
# Duplicate a handful of orders into a late-arriving file (Lab 2 dedup exercise)
import json, random
src = [json.loads(l) for l in open(f"{VOL}/orders/batch_01.json")][:25]
for r in src:
    r["amount"] = f"{float(r['amount'] or 0) + 10:.2f}"   # corrected amounts
    r["order_ts"] = "2026-04-01 09:00:00"                 # later ingest
with open(f"{VOL}/orders_late.json", "w") as f:
    for r in src:
        f.write(json.dumps(r) + "\n")
print("late-arriving corrections:", len(src))

---
# Lab 1 — Ingestion (3h)

**Objectives:** S2 — Auto Loader with schema enforcement and evolution; `COPY INTO`
incremental loading; choosing between them.

**Burn in:**
- The four `schemaEvolutionMode` values and what each does when a new column appears
- That `addNewColumns` (the default) *fails the stream* and recovers on restart
- That `COPY INTO` tracks loaded files and silently skips them on re-run
- Where `_rescued_data` comes from

### 1a — Auto Loader, default evolution mode

In [0]:
bronze_path = f"{VOL}/_schemas/bronze_orders"
ckpt        = f"{VOL}/_ckpt/bronze_orders"

(spark.readStream
   .format("cloudFiles")
   .option("cloudFiles.format", "json")
   .option("cloudFiles.schemaLocation", bronze_path)
   .load(f"{VOL}/orders")
 .writeStream
   .option("checkpointLocation", ckpt)
   .option("mergeSchema", "true")          # <-- this was missing
   .trigger(availableNow=True)
   .toTable("bronze_orders"))

**Observe:** `trigger(availableNow=True)` processes everything available then stops —
the modern replacement for `trigger(once=True)`. Note it in your scratch cell; the
exam may still show `once`.

In [0]:
spark.sql("SELECT count(*) AS rows FROM bronze_orders").display()
spark.sql("DESCRIBE bronze_orders").display()

### 1b — Land the new column

Now write `batch_03`, which has a `channel` field the stream has never seen, and
re-run the stream cell above.

In [0]:
print("batch_03:", make_batch("batch_03.json", 300, 3000, with_channel=True))

**Now re-run cell 1a.** Expected: it **fails** with `UnknownFieldException`.
That is `addNewColumns` behaving correctly — it records the new schema, then fails.

**Run 1a a second time.** It now succeeds and `channel` is present.

> This is the single most-missed Auto Loader fact. The default mode *does* fail;
> it just self-heals on restart. Under a job with retries that's invisible. Under a
> continuous SLA it's an outage.

In [0]:
spark.sql("DESCRIBE bronze_orders").display()   # channel should now appear

### 1c — `rescue` mode, on a fresh stream

Same source, new target, so you can compare behaviour side by side.

In [0]:
(spark.readStream
   .format("cloudFiles")
   .option("cloudFiles.format", "json")
   .option("cloudFiles.schemaLocation", f"{VOL}/_schemas/bronze_rescue")
   .option("cloudFiles.schemaEvolutionMode", "rescue")
   .option("cloudFiles.schemaHints", "order_id STRING, amount STRING")
   .load(f"{VOL}/orders")
 .writeStream
   .option("checkpointLocation", f"{VOL}/_ckpt/bronze_rescue")
   .option("mergeSchema", "true")          # <-- this was missing
   .trigger(availableNow=True)
   .toTable("bronze_rescue"))

spark.sql("SELECT _rescued_data FROM bronze_rescue WHERE _rescued_data IS NOT NULL LIMIT 5").display()

**Observe:** the stream never failed, and unexpected fields landed in `_rescued_data`
as JSON. That's the answer whenever a stem says *the stream must not stop*.

**Try:** re-run with `failOnNewColumns` and with `none` against fresh targets.
Four modes, four behaviours — write them from memory afterwards before checking.

##### Trying again with failOnNewColumns setting

In [0]:
# move batch_03 out of orders/
dbutils.fs.mv(f"{VOL}/orders/batch_03.json", f"{VOL}/_held/batch_03.json")

In [0]:
(spark.readStream
   .format("cloudFiles")
   .option("cloudFiles.format", "json")
   .option("cloudFiles.schemaLocation", f"{VOL}/_schemas/bronze_failnew") # <- edited: bronze_rescue -> bronze_failnew
   .option("cloudFiles.schemaEvolutionMode", "failOnNewColumns")
   .option("cloudFiles.schemaHints", "order_id STRING, amount STRING")
   .load(f"{VOL}/orders")
 .writeStream
   .option("checkpointLocation", f"{VOL}/_ckpt/bronze_failnew") # <- edited: bronze_rescue -> bronze_failnew
   .option("mergeSchema", "true")          # <-- this was missing
   .trigger(availableNow=True)
   .toTable("bronze_failnew")) # <- edited: bronze_rescue -> bronze_failnew

spark.sql("DESCRIBE bronze_failnew").display()

In [0]:
# move batch3 back to its initial location
dbutils.fs.mv(f"{VOL}/_held/batch_03.json", f"{VOL}/orders/batch_03.json")

Rerunning the write / read cell is then expected to fail on all attempts... which it does.

##### Trying again with evolveSchema setting 'none'

In [0]:
# move batch_03 out of orders/
dbutils.fs.mv(f"{VOL}/orders/batch_03.json", f"{VOL}/_held/batch_03.json")

In [0]:
display(dbutils.fs.ls(f"{VOL}/orders"))

In [0]:
(spark.readStream
   .format("cloudFiles")
   .option("cloudFiles.format", "json")
   .option("cloudFiles.schemaLocation", f"{VOL}/_schemas/bronze_fail_evolve_none") 
   .option("cloudFiles.schemaEvolutionMode", "none")
   .option("cloudFiles.schemaHints", "order_id STRING, amount STRING")
   .load(f"{VOL}/orders")
 .writeStream
   .option("checkpointLocation", f"{VOL}/_ckpt/bronze_fail_evolve_none") 
   .option("mergeSchema", "true")          
   .trigger(availableNow=True)
   .toTable("bronze_fail_evolve_none")) 

spark.sql("DESCRIBE bronze_fail_evolve_none").display()

In [0]:
spark.sql("select * from bronze_fail_evolve_none limit 10").display()

In [0]:
# move batch3 back to its initial location
dbutils.fs.mv(f"{VOL}/_held/batch_03.json", f"{VOL}/orders/batch_03.json")

### 1d — `COPY INTO` and its file tracking

In [0]:
spark.sql("DROP TABLE IF EXISTS bronze_copy")
spark.sql("CREATE TABLE bronze_copy (order_id STRING, customer_id STRING, order_ts STRING, amount STRING, region STRING)")

spark.sql(f'''
COPY INTO bronze_copy
FROM (SELECT order_id, customer_id, order_ts, amount, region FROM '{VOL}/orders')
FILEFORMAT = JSON
''').display()

In [0]:
# Run the SAME statement again — watch num_affected_rows
spark.sql(f'''
COPY INTO bronze_copy
FROM (SELECT order_id, customer_id, order_ts, amount, region FROM '{VOL}/orders')
FILEFORMAT = JSON
''').display()

spark.sql("SELECT count(*) FROM bronze_copy").display()

**Observe:** second run loads **zero** rows. `COPY INTO` remembers which files it
has ingested — that's the idempotency guarantee, and it's file-level, not row-level.

**The exam's discriminator:** dozens/thousands of files and simple batch → `COPY INTO`.
Millions of files, evolving schema, or continuous arrival → Auto Loader, and
**file notification mode** specifically when directory listing has become the bottleneck.

*(File notification mode needs cloud event configuration you can't do on Free Edition —
know what it solves and why, you won't be asked to configure it.)*

In [0]:
# SCRATCH — Lab 1. What surprised you?

---
# Lab 2 — Bronze → Silver (2h)

**Objectives:** S3 status-1 — cleaning nulls and standardizing types; joins.
S3 status-2 — deduplication and aggregation; column/row manipulation.

**Burn in:**
- Non-ANSI casting returns `null` rather than raising (so empty strings are safe)
- `dropDuplicates` keeps an **arbitrary** row; a window function keeps the one you chose
- `explode` drops rows with empty arrays; `explode_outer` keeps them
- Two `explode`s in one SELECT produce a **cross product**

### 2a — Type standardisation

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("bronze_orders")

silver = (bronze
    .withColumn("order_ts", F.col("order_ts").cast("timestamp"))
    .withColumn("amount",   F.col("amount").cast("double")))

silver.select("order_id","order_ts","amount").show(5, truncate=False)
print("null amounts (were empty strings):",
      silver.filter(F.col("amount").isNull()).count())

**Observe:** no exception. The empty strings became `null`.

**Try:** `spark.conf.set("spark.sql.ansi.enabled", True)` and re-run.
ANSI mode makes Spark *stricter* — the cast now raises. Set it back to `False` after.
This is the fact behind a distractor that claims ANSI coerces bad values to zero. It doesn't.

### 2b — Deduplication: arbitrary vs deliberate

In [0]:
# Union the late-arriving corrections in, so each order_id may appear twice
late = (spark.read.json(f"{VOL}/orders_late.json")
          .withColumn("order_ts", F.col("order_ts").cast("timestamp"))
          .withColumn("amount",   F.col("amount").cast("double")))

combined = silver.select("order_id","customer_id","order_ts","amount","region").unionByName(
           late.select("order_id","customer_id","order_ts","amount","region"))

print("rows:", combined.count(), "| distinct order_ids:", combined.select("order_id").distinct().count())

In [0]:
# Approach 1: dropDuplicates — which row survives?
d1 = combined.dropDuplicates(["order_id"])
d1.filter(F.col("order_id")=="o1000").select("order_id","order_ts","amount").show()

In [0]:
# Run the SAME cell a few times, and/or repartition first. Does the surviving row change?
combined.repartition(7).dropDuplicates(["order_id"]) \
        .filter(F.col("order_id")=="o1000").select("order_id","order_ts","amount").show()

In [0]:
# Approach 2: window function — latest wins, deterministically
from pyspark.sql.window import Window

w = Window.partitionBy("order_id").orderBy(F.col("order_ts").desc())
d2 = (combined.withColumn("rn", F.row_number().over(w))
              .filter(F.col("rn")==1).drop("rn"))

d2.filter(F.col("order_id")=="o1000").select("order_id","order_ts","amount").show()

**Observe:** `dropDuplicates` gave you *a* row per key with no control over which.
The window function gave you the latest, every time.

> Stem language to map: *"most recent"*, *"latest version"*, *"keep the corrected record"*
> → window function. Bare *"remove duplicates"* with identical rows → `DISTINCT` /
> `dropDuplicates`. Incremental upsert into an existing table → `MERGE` with **both**
> `WHEN MATCHED` and `WHEN NOT MATCHED`.

### 2c — Nested data: explode

In [0]:
nested = spark.table("bronze_orders").select("order_id","items")
nested.printSchema()

# Correct: explode once, then project off the struct
flat = nested.select("order_id", F.explode("items").alias("item")) \
             .select("order_id", "item.sku", "item.qty")
flat.show(5)
print("order rows:", nested.count(), "-> line rows:", flat.count())

In [0]:
# WRONG on purpose: two explodes in one select = cross product
bad = nested.select("order_id",
                    F.explode("items.sku").alias("sku"),
                    F.explode("items.qty").alias("qty"))
print("cross-product rows:", bad.count(), " (should be far more than", flat.count(), ")")
bad.filter(F.col("order_id")==nested.first()["order_id"]).show()

In [0]:
# explode vs explode_outer on an empty array
from pyspark.sql import Row
t = spark.createDataFrame([Row(id="a", arr=[1,2]), Row(id="b", arr=[])])
print("explode:")       ; t.select("id", F.explode("arr")).show()
print("explode_outer:") ; t.select("id", F.explode_outer("arr")).show()

**Observe:** order `b` vanished under `explode` and survived under `explode_outer`.

Also note `items.sku` on an `ARRAY<STRUCT>` returns an **array of skus** — one row,
still nested. That's the distractor shape: dot notation works on a plain `STRUCT`,
not on an array of them.

### 2d — Write silver, then join

In [0]:
(d2.write.mode("overwrite").saveAsTable("silver_orders"))

# Left join keeps every order even where no line items exist
lines = flat.groupBy("order_id").agg(F.sum("qty").alias("total_qty"))
joined = spark.table("silver_orders").join(lines, "order_id", "left")

print("silver:", spark.table("silver_orders").count(), "| after left join:", joined.count())
print("no line items:", joined.filter(F.col("total_qty").isNull()).count())

In [0]:
# Semi and anti joins — the existence filters
o = spark.table("silver_orders")
print("semi (has lines):", o.join(lines, "order_id", "left_semi").count())
print("anti (no lines):",  o.join(lines, "order_id", "left_anti").count())

In [0]:
# SCRATCH — Lab 2. What surprised you?

---
# Lab 3 — Gold layer objects (1.5h)

**Objective:** S3 **status-1** — the difference between materialized views, views,
streaming tables and tables, and when each is right.

This is a status-1 objective and it cost you a mark on the mock. Build the same
aggregate four ways and compare behaviour when the source changes.

**[MAY NOT RUN ON FREE EDITION]** — `CREATE MATERIALIZED VIEW` and `CREATE STREAMING TABLE`
in a notebook may require a SQL warehouse or a Lakeflow pipeline. If the cell errors,
run it from a SQL editor cell (`%sql`) against a serverless SQL warehouse, or read the
comparison table below and move on — the *semantics* are what's examined, not the DDL.

In [0]:
# 1. VIEW — logic only, nothing stored
spark.sql('''
CREATE OR REPLACE VIEW gold_by_region_view AS
SELECT region, count(*) AS orders, round(sum(amount),2) AS revenue
FROM silver_orders GROUP BY region
''')
spark.sql("SELECT * FROM gold_by_region_view ORDER BY region").display()

In [0]:
# 2. TABLE — you own the refresh
spark.sql('''
CREATE OR REPLACE TABLE gold_by_region_table AS
SELECT region, count(*) AS orders, round(sum(amount),2) AS revenue
FROM silver_orders GROUP BY region
''')
spark.sql("SELECT * FROM gold_by_region_table ORDER BY region").display()

In [0]:
# 3. MATERIALIZED VIEW — precomputed, incrementally refreshed
spark.sql('''
CREATE OR REPLACE MATERIALIZED VIEW gold_by_region_mv AS
SELECT region, count(*) AS orders, round(sum(amount),2) AS revenue
FROM silver_orders GROUP BY region
''')
spark.sql("SELECT * FROM gold_by_region_mv ORDER BY region").display()

In [0]:
# Now mutate the source and re-query all three WITHOUT refreshing anything
spark.sql("INSERT INTO silver_orders SELECT * FROM silver_orders LIMIT 50")

for t in ["gold_by_region_view","gold_by_region_table","gold_by_region_mv"]:
    print("---", t)
    try:
        spark.sql(f"SELECT sum(orders) AS total_orders FROM {t}").show()
    except Exception as e:
        print("  (unavailable here):", str(e)[:120])

**Observe:** the view moved immediately. The table did not. The MV did not until refreshed.
That is the whole distinction, and it's what the exam is asking when it gives you a
freshness tolerance and a query-volume figure.

| Object | Stored? | Freshness | Cost falls on | Use when |
|---|---|---|---|---|
| View | No | Always current | Every read | Cheap logic, low query volume |
| Materialized view | Yes | As of last refresh | Refresh | **BI aggregates** — precomputed, incrementally maintained |
| Streaming table | Yes | Continuous/incremental | Each update | Append-only, low-latency ingestion |
| Table | Yes | As of last write | Your job | You control the write logic |

**Exam trigger words:** *sub-second BI* + *tolerates N minutes stale* → materialized view.
*Append-only, process each row once* → streaming table. *Always current, low volume* → view.

In [0]:
# 4. STREAMING TABLE — run if your compute supports it
spark.sql(f'''
CREATE OR REFRESH STREAMING TABLE gold_stream_orders
AS SELECT * FROM STREAM(silver_orders)
''')

**If that failed:** expected on some compute. The fact to hold instead — a streaming
table processes each source row **once**, tracked by checkpoint, so the first update
consumes history and later updates handle only new rows. A **full refresh** discards
the checkpoint and reprocesses everything; that's what you need after changing
transformation logic. Source must be **append-only** or the stream fails.

In [0]:
# SCRATCH — Lab 3. What surprised you?

---
# Lab 4 — Governance (2h)

**Objectives:** S7 **status-1** — column masking and row-level security.
S7 status-2 — GRANT/REVOKE/DENY across the hierarchy. Plus managed vs external tables.

**Burn in:**
- **Row filter** = which rows. **Column mask** = what a value looks like. (You swapped these on the mock.)
- Managed vs external: what `DROP` does to the files
- `SELECT` is inert without `USE CATALOG` + `USE SCHEMA`

**Free Edition note:** no account console means no account-level groups, and UC `GRANT`
cannot use workspace-level groups. The labs below use `current_user()` instead of
`is_account_group_member()`. **The mechanism is identical — only the predicate differs.**
Remember that on the exam the group form is what you'll see.

### 4a — Managed vs external

In [0]:
spark.sql("CREATE OR REPLACE TABLE t_managed AS SELECT * FROM silver_orders LIMIT 100")
spark.sql("DESCRIBE EXTENDED t_managed").display()

**Observe:** find `Type` (MANAGED) and `Location`. The location is metastore-managed
storage — you never specified it. That's the whole definition.

**The consequence to memorise:**

| | Managed | External |
|---|---|---|
| Location | Metastore managed storage | You specify `LOCATION` |
| `DROP TABLE` | metadata **and data files** deleted | metadata only; **files remain** |
| Predictive optimization | Fully supported | Limited |
| Default choice | Yes | Only when another system also owns the files |

Conversion between the two **is** supported — the outline lists it explicitly, so any
option claiming it's impossible is wrong by construction.

In [0]:
# External table on the volume (works on Free Edition since volumes are UC-governed)
spark.sql(f"DROP TABLE IF EXISTS t_external")
spark.sql(f'''
CREATE TABLE t_external
LOCATION '{VOL}/_ext/t_external'
AS SELECT * FROM silver_orders LIMIT 100
''')
spark.sql("DESCRIBE EXTENDED t_external").display()

In [0]:
# Drop the external table, then check the files are still there
spark.sql("DROP TABLE t_external")
try:
    display(dbutils.fs.ls(f"{VOL}/_ext/t_external"))
    print(">>> Files survive the DROP. That's the external-table contract.")
except Exception as e:
    print(e)

### 4b — Column mask

In [0]:
spark.sql('''
CREATE OR REPLACE FUNCTION mask_amount(v DOUBLE)
RETURN CASE
  WHEN current_user() = session_user() THEN v   -- swap for is_account_group_member('finance')
  ELSE NULL
END
''')

spark.sql("CREATE OR REPLACE TABLE masked_orders AS SELECT * FROM silver_orders")
spark.sql("ALTER TABLE masked_orders ALTER COLUMN amount SET MASK mask_amount")

spark.sql("SELECT order_id, region, amount FROM masked_orders LIMIT 5").display()

In [0]:
spark.sql("DESCRIBE EXTENDED masked_orders").display()   # find the mask on `amount`

**Observe:** every row is still there. Only the *value* changed. That is a column mask,
and it's why a mask is the wrong tool when the requirement is to hide **rows**.

### 4c — Row filter

In [0]:
spark.sql('''
CREATE OR REPLACE FUNCTION emea_only(r STRING)
RETURN r = 'EMEA' OR current_user() = session_user()
''')

spark.sql("CREATE OR REPLACE TABLE filtered_orders AS SELECT * FROM silver_orders")
spark.sql("ALTER TABLE filtered_orders SET ROW FILTER emea_only ON (region)")

spark.sql("SELECT region, count(*) FROM filtered_orders GROUP BY region").display()

**Observe:** the filter is a **predicate**, so non-matching rows do not exist for that
principal — not nulled, absent. Flip the function to `RETURN r = 'EMEA'` and re-query
to see the filtered view.

> **Row filter → which rows. Column mask → what the value looks like.**
> Write that sentence out. It's ~2 marks.

**ABAC** is the layer above both: tag columns, write **one** policy keyed on the tag,
and it governs every table carrying it — including tables created later. Stem language:
*"defined once"*, *"consistently across many tables"* → ABAC, not per-table attachment.

### 4d — Privilege hierarchy

In [0]:
spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOG}.{SCHEMA}").display()
spark.sql("SHOW GRANTS ON TABLE silver_orders").display()

**The model (memorise, you can't fully exercise it solo):**

- Three levels: `USE CATALOG` → `USE SCHEMA` → `SELECT`. A grant on the table is
  **inert** without traversal rights above it. *This is the most common UC permission error.*
- Privileges **inherit downward** — `SELECT` on a schema covers all its tables, present and future.
- `DENY` **overrides** any inherited grant. It's for carving exceptions out of a broad grant.
- `DENY` is **never** a default — new principals simply have nothing.
- **Ownership** is separate from privileges. `ALL PRIVILEGES` ≠ ownership.
- Grant to **groups**, not users.
- Legacy `USAGE` → now `USE CATALOG` / `USE SCHEMA`. Bare `USAGE` in an option = stale phrasing.

In [0]:
# SCRATCH — Lab 4. What surprised you?

---
# Lab 5 — Lakeflow Jobs control flow (2h)

**Objectives:** S4 status-2 — control flows (retries, branching, looping);
task configuration and dependencies via the DAG.

Built in the **Jobs & Pipelines** UI, not here. This notebook holds the task code.
Run Lab 0 first so the tables exist.

## What to build

A job named `de_lab_job` with five tasks:

```
          ┌──> publish ──┐
validate ─┤              ├──> notify_ops
          └──> (skipped) ┘
                 ▲
   ingest ───────┘
   for_each_region  (parallel, 3 iterations)
```

| Task | Type | Config |
|---|---|---|
| `ingest` | Notebook → cell 5a | — |
| `validate` | Notebook → cell 5b | **Retries: 3**, interval 30s |
| `check_threshold` | **If/else condition** | `{{tasks.validate.values.row_count}}` `>` `100` |
| `publish` | Notebook → cell 5c | depends on `check_threshold` (**true** branch) |
| `for_each_region` | **For each** | Inputs `["EMEA","AMER","APAC"]`, concurrency 2, nested task → cell 5d |
| `notify_ops` | Notebook → cell 5e | depends on `publish`, run-if **"At least one failed"** |

Job-level settings: **timeout** 1h · **max concurrent runs = 1** · failure notification to your email.

### 5a — `ingest` task code

In [0]:
row_count = spark.table("bronze_orders").count()
print("bronze rows:", row_count)
dbutils.jobs.taskValues.set(key="bronze_rows", value=row_count)

### 5b — `validate` task code (set the task value the condition reads)

In [0]:
n = spark.table("silver_orders").count()
print("silver rows:", n)
dbutils.jobs.taskValues.set(key="row_count", value=n)

# To exercise the retry policy, uncomment: fails twice, then succeeds
# import random
# if random.random() < 0.6:
#     raise RuntimeError("simulated flaky upstream")

### 5c — `publish` task code

In [0]:
spark.sql('''
CREATE OR REPLACE TABLE gold_published AS
SELECT region, count(*) AS orders, round(sum(amount),2) AS revenue
FROM silver_orders GROUP BY region
''')
print("published")

### 5d — nested task inside `For each` (parameter name: `region`)

In [0]:
region = dbutils.widgets.get("region")
n = spark.table("silver_orders").filter(f"region = '{region}'").count()
print(f"{region}: {n} rows")

### 5e — `notify_ops` task code

In [0]:
print("ALERT: publish did not succeed")

## What to observe — this is the lab, not the building

1. **Run it normally.** `notify_ops` shows **Skipped**, and the run is **Succeeded**.
   A skipped task does not fail a run. *(This exact signature was a mock question.)*
2. **Break `publish`** (add `raise RuntimeError("boom")`). Re-run.
   `notify_ops` now runs; the run is Failed.
3. **Uncomment the flaky block in `validate`.** Watch the run history show
   attempt 1, 2, 3 under one task — not three job runs.
4. **Set the If/else threshold above your row count.** `publish` and everything
   downstream go **Skipped**, run still **Succeeded**.
5. **Trigger the job twice quickly** with max concurrent runs = 1. Second is queued/skipped.

## The two settings people conflate

| | Scope | Purpose |
|---|---|---|
| **Retry policy** | Task (or job) | Re-run after *transient* failure. Never fixes data errors. |
| **Run-if condition** | Dependency edge | Whether a task runs given upstream **outcomes** |

Run-if values: *All succeeded* (default) · At least one succeeded · None failed ·
All done · **At least one failed** · All failed.

Task **values** carry data; **If/else** branches on that data; **run-if** reacts to outcomes.

## Triggers — configure each once, then delete

| Trigger | Fires when | Use for |
|---|---|---|
| Scheduled (cron) | A time arrives | Predictable cadence |
| **File arrival** | New files at a monitored path | Irregular external drops |
| **Table update** | A monitored UC table is written | Cross-job dependencies you don't own |
| Continuous | — runs perpetually | Lowest latency, highest cost |

Point a file-arrival trigger at `/Volumes/.../landing/orders` and drop a file in.

---
# Lab 6 — Declarative Automation Bundles (2.5h)

**Objectives:** S5 status-1 — Git Folders workflow. S5 status-2 — environment-specific
configuration with bundle variables and overrides.

**Run these in a terminal on your laptop, not in this notebook.**
This is the highest-value block in the pack: three of your six CI/CD mock misses were
**invented CLI flags**, and that error mode disappears once you've typed the real ones.

## Setup

```bash
# macOS/Linux
curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh
databricks --version
databricks auth login --host https://<your-workspace>.cloud.databricks.com
```

## The lab

```bash
databricks bundle init            # pick default-python
cd <project>

databricks bundle validate                 # <- the CI gate. No workspace changes.
databricks bundle validate -t dev
databricks bundle deploy -t dev
databricks bundle run <job_name> -t dev
databricks bundle summary -t dev
```

Then **go look in the workspace UI** at Jobs & Pipelines.

## Five things to observe

1. The deployed job is named `[dev <yourname>] ...` — **development mode prefixing**.
2. Its schedule is **Paused**. Development mode pauses schedules and triggers.
   *(A deployed job that never fires + a dev target = expected behaviour, not a bug.)*
3. Add a variable and override it per target in `databricks.yml`:

   ```yaml
   variables:
     catalog:
       description: Target catalog
       default: dev_analytics

   targets:
     dev:
       mode: development
       default: true
     prod:
       mode: production
       variables:
         catalog: prod_analytics
       run_as:
         service_principal_name: <app-id>
   ```

   Run `databricks bundle validate -t prod` and find the resolved value.
4. **Run `databricks bundle deploy` with no `-t`.** It goes to the **default target**,
   not prod. This is how prod deploys quietly land in dev. *Always pass `-t` in CI.*
5. **Break the YAML** — bad indent, undefined variable reference, unknown key.
   Run `validate` and read each error. This is the inoculation against fake flags.

## The command surface — there is nothing else

| Command | Effect |
|---|---|
| `bundle init` | Scaffold from a template |
| `bundle validate` | Parse + check config; **no** workspace changes |
| `bundle deploy` | Create/update resources in the target |
| `bundle run` | Execute a named job or pipeline |
| `bundle summary` | Show what's deployed to a target |
| `bundle destroy` | Remove deployed resources |

There is **no** `--dry-run`, no `--resources`, no `--validate-only`.
If an exam option shows a flag you didn't type this week, that's evidence against it.

## Development vs production mode

| | development | production |
|---|---|---|
| Resource prefix | `[dev user]` | none |
| Schedules/triggers | **Paused** | Active |
| Deploy path | Your user folder | Shared path |
| `run_as` | Deploying user | Set explicitly (service principal) |

## Git Folders (status-1) — 20 minutes

In the workspace: Workspace → Create → Git folder → clone your repo.
Create a branch, edit a notebook, commit and push — **all from the UI**.
Then click through to create the PR.

**The boundary that gets examined:** Git Folders do branch/commit/push/pull and
*link out* to create a PR. **Review, approval and merge happen in the Git provider.**
Any option offering "merge from the Git Folder UI" is wrong.

---
# Lab 7 — Monitoring (1h)

**Objectives:** S6 status-2 — run-history trends; Jobs UI health and DAG reading.

Costs almost nothing because Labs 1–6 generated the material. Nothing to build.

## Walk these four screens

1. **Jobs → `de_lab_job` → Runs.** Compare durations across your runs. This is
   literally the outline's wording: *compare current execution times against historical baselines*.
2. **Open a failed run → the DAG.** Find a task showing **Upstream failed** —
   that tells you to look **up** the graph, not at the task.
3. **Open a task → Spark UI → Stages.** In the task summary find Min / Median / Max
   shuffle read. Learn where that table lives; you'll be asked to interpret it blind.
4. **Task detail → the retry attempts** from Lab 5.

## The diagnostic ladder

**Job run history → task DAG → Spark UI stage metrics.** Narrow before you drill.

| Symptom in stage metrics | Diagnosis | Fix |
|---|---|---|
| Max shuffle read ≫ median; one straggler | **Skew** | AQE skew join, salting, broadcast |
| Uniform task durations, heavy spill | **Volume per partition** | ↑ `spark.sql.shuffle.partitions` |
| Large shuffle both sides, one side small | **Missed broadcast** | ↑ `autoBroadcastJoinThreshold` |
| **One task only, no scaling with cluster size** | **Unsplittable input** | more files / splittable codec |

That last row is your repeat miss. **Read parallelism ≠ shuffle parallelism.**
`spark.sql.shuffle.partitions` governs partitions *after* a wide transformation;
it has no effect on how an input file is split.

| Codec / format | Splittable |
|---|---|
| gzip | **No** |
| snappy on raw CSV/JSON | No |
| bzip2, lz4, zstd | Yes |
| Parquet / ORC / Avro | Yes (snappy *inside* them is fine) |

## What the tools answer

| Tool | Question |
|---|---|
| Run history | Is this getting slower? Which task regressed? |
| Jobs UI / DAG | Current status? Which upstream is blocking? Failure rate? |
| Spark UI stages | Within a slow task: skew, shuffle, or spill? |
| `DESCRIBE HISTORY` | What **changed** in this table, when, by whom (writes only) |
| System tables (`system.access.audit`) | Who **read** it, and which columns |

`DESCRIBE HISTORY` has **no read entries**. Read auditing = UC audit logs / system tables.

**Liquid Clustering** replaces partitioning + ZORDER; keys changeable without rewrite;
suits high-cardinality and shifting query patterns.
**Predictive optimization** auto-runs OPTIMIZE/VACUUM/stats on UC **managed** tables —
it is maintenance, **not** monitoring and **not** alerting.

In [0]:
spark.sql("DESCRIBE HISTORY silver_orders").display()   # writes only — no reads here

---
# Daily Drill — Lakeflow Connect taxonomy

**15 minutes at the start of each session, before you touch a lab.**
This is the block you cannot practise hands-on — you can't provision Salesforce or a
SQL Server CDC gateway in eight days. It is the densest new material on the exam
and it must be memorised on a schedule.

Fill these in **from memory**, then check against the key at the very bottom.
Spaced retrieval across 8 days beats one long block.

### Table A — components by connector type

| Connector type | Example sources | Components required |
|---|---|---|
| Managed SaaS | | |
| Managed database (CDC) | | |
| Query-based | | |
| Managed streaming | | |
| Managed file | | |
| Standard | | |
| Community / custom | | |

### Table B — the selection ladder

Write the four rungs, top to bottom, and the rule for dropping a rung:

1.
2.
3.
4.

Rule: ______________________________________________

### Table C — Auto Loader schema evolution

| Mode | Behaviour on a new column |
|---|---|
| `addNewColumns` (default) | |
| `rescue` | |
| `failOnNewColumns` | |
| `none` | |

### Table D — `COPY INTO` vs Auto Loader

| | `COPY INTO` | Auto Loader |
|---|---|---|
| Scale | | |
| Mechanism | | |
| File discovery | | |
| Schema evolution | | |
| Choose when | | |

### Table E — one-liners

- Row filter vs column mask: ______________________
- ABAC is for: ____________________________________
- Managed vs external on `DROP`: __________________
- Dev vs prod bundle mode: ________________________
- Run-if "at least one failed" is for: _____________

---
# Rename map — five free marks

Distractors are built from the **old** names.

| Old | New (May 2026) |
|---|---|
| Delta Live Tables (DLT) | Lakeflow Spark Declarative Pipelines |
| Databricks Workflows | Lakeflow Jobs |
| Databricks Repos | Databricks Git Folders |
| Databricks Asset Bundles | Declarative Automation Bundles (still "DAB") |
| SQL endpoint | SQL warehouse |
| High-concurrency cluster | Serverless SQL warehouse |
| `USAGE` | `USE CATALOG` / `USE SCHEMA` |
| `trigger(once=True)` | `trigger(availableNow=True)` |

---
# Your three failure modes (from the 45-question mock)

**1. Fabricated options** — `--resources`, `--dry-run`, `cloudFiles.format = "rest"`.
Four marks. Before judging whether an option's behaviour is *desirable*, check the
tool's **domain**: Auto Loader reads object storage, full stop. Condition tasks read
task values. Bundles don't talk to Git providers. Lab 6 is the cure.

**2. Mechanism substitution** — mask for filter, `flatten` for `explode`, SaaS
components for database components. Concept right, artefact swapped. Labs 2 and 4
plus the daily drill.

**3. Read vs shuffle parallelism** — Lab 7's table. One task and no scaling = input
splittability. Even tasks and spill = shuffle partitions. One straggler = skew.

**Also:** options containing *never* / *always* / *cannot under any circumstances* are
wrong far more often than chance, especially in governance where capability keeps expanding.

---
---
# ⚠️ ANSWER KEY — don't scroll here until you've written the drill from memory
---
---

### Table A
| Connector type | Example sources | Components |
|---|---|---|
| Managed SaaS | Salesforce, Workday, ServiceNow | Connection + ingestion pipeline + destination tables |
| Managed database (CDC) | SQL Server, MySQL, PostgreSQL | Connection + **ingestion gateway** + **staging storage** + pipeline |
| Query-based | Databases, scheduled polling | Connection + pipeline (**no** gateway, **no** staging) |
| Managed streaming | Message buses | Connection + pipeline → streaming tables |
| Managed file | SharePoint, Google Drive | Connection + pipeline |
| Standard | Cloud object storage; Kafka/Kinesis/Pub-Sub/Pulsar | You configure — Structured Streaming (full control) **or** Lakeflow pipelines (more managed) |
| Community / custom | Unsupported sources | You build; no Databricks SLA |

### Table B
1. Managed connector → 2. Standard connector → 3. Auto Loader / `COPY INTO` →
4. Custom code (REST/JDBC in a notebook) orchestrated by Lakeflow Jobs.

Rule: **start at the most managed layer; drop a rung only when it can't reach your
source or meet a processing requirement** (e.g. custom binary payload parsing).

### Table C
| Mode | Behaviour |
|---|---|
| `addNewColumns` (default) | Stream **fails**, schema updated, succeeds on restart |
| `rescue` | Never fails; unknown fields → `_rescued_data` |
| `failOnNewColumns` | Fails and stays failed until schema updated manually |
| `none` | New columns ignored |

### Table D
| | `COPY INTO` | Auto Loader |
|---|---|---|
| Scale | Thousands of files, low frequency | Millions, high frequency |
| Mechanism | SQL command; tracks loaded files | Streaming source (`cloudFiles`) + checkpoint |
| File discovery | Lists directory each run | Directory listing **or** file notification |
| Schema evolution | Manual | Built in via `schemaLocation` |
| Choose when | Simple bounded idempotent batch | Volume, evolving schema, continuous |

### Table E
- **Row filter → which rows. Column mask → what the value looks like.**
- ABAC: tag-driven policy, **defined once**, applied across many tables incl. future ones.
- `DROP`: managed deletes metadata **and files**; external deletes metadata only.
- dev = `[dev user]` prefix + **paused** schedules; prod = no prefix, active, explicit `run_as`.
- Failure notification / cleanup tasks that must run **only** when something failed.